# Accelerated Gradient Methods

**S&DS 432/632 -- Advanced Optimization Techniques**

This notebook accompanies Chapter 9 (Accelerated Gradient Methods) of the lecture notes. It contains:

1. **GD vs. Nesterov's AGD** -- convergence comparison on a quadratic ($\kappa = 500$), using the exact $\theta_k$ recursion
2. **FISTA for Lasso** -- accelerated proximal gradient for composite optimization
3. **Momentum visualization** -- iterate trajectories on an ill-conditioned quadratic
4. **Strongly convex case** -- GD vs. heavy-ball vs. Nesterov AGD-SC, plus condition number dependence
5. **Adaptive restart** -- gradient-based restart for recovering linear convergence without knowing $\mu$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Morandi color palette
TERRACOTTA = '#C47A6A'
SAGE       = '#7A9A7E'
STEEL_BLUE = '#7B97AD'
GOLD       = '#B8995E'
LAVENDER   = '#907EA0'

## 1. GD vs. Nesterov's Accelerated Gradient

Gradient descent on an $L$-smooth convex function converges at rate $O(1/k)$. Nesterov's accelerated gradient method improves this to $O(1/k^2)$ by adding a **momentum** (extrapolation) step. The algorithm maintains two sequences with the **exact $\theta_k$ recursion**:

$$
\begin{aligned}
y^{k+1} &= x^k - \frac{1}{L}\nabla f(x^k), \\
\theta_{k+1} &= \frac{1 + \sqrt{1 + 4\theta_k^2}}{2}, \quad \theta_0 = 1, \\
x^{k+1} &= y^{k+1} + \frac{\theta_k - 1}{\theta_{k+1}}\bigl(y^{k+1} - y^k\bigr).
\end{aligned}
$$

The momentum coefficient $\beta_k = (\theta_k - 1)/\theta_{k+1}$ increases toward $1$ as $k \to \infty$. For large $k$, $\beta_k \approx k/(k+3)$, but the exact recursion is what makes the convergence proof work (via the identity $\theta_k(\theta_k - 1) = \theta_{k-1}^2$).

**Convergence guarantee (Theorem 11.4):** For convex, $L$-smooth $f$,
$$
f(y^k) - f(x^*) \leq \frac{2L\|x^0 - x^*\|_2^2}{(k+1)^2}.
$$

### Test problem: Quadratic with large condition number

We minimize $f(x) = \frac{1}{2} \sum_{i=1}^n \lambda_i x_i^2$ with eigenvalues uniformly spaced in $[\mu, L]$, giving condition number $\kappa = L/\mu = 500$.

In [ ]:
def gradient_descent(Q, x0, num_iters):
    """Gradient descent on f(x) = 0.5 * x^T Q x with stepsize 1/L."""
    L = np.max(Q)
    x = x0.copy()
    f_vals = [0.5 * x @ (Q * x)]
    for _ in range(num_iters):
        grad = Q * x
        x = x - (1.0 / L) * grad
        f_vals.append(0.5 * x @ (Q * x))
    return np.array(f_vals)


def nesterov_ag(Q, x0, num_iters):
    """Nesterov's accelerated gradient (convex case, exact theta_k recursion)."""
    L = np.max(Q)
    x = x0.copy()
    y = x0.copy()
    theta = 1.0
    f_vals = [0.5 * y @ (Q * y)]
    for k in range(num_iters):
        grad = Q * x
        y_new = x - (1.0 / L) * grad
        theta_new = (1 + np.sqrt(1 + 4 * theta**2)) / 2
        momentum = (theta - 1) / theta_new
        x_new = y_new + momentum * (y_new - y)
        y = y_new
        x = x_new
        theta = theta_new
        f_vals.append(0.5 * y @ (Q * y))
    return np.array(f_vals)

In [ ]:
# Problem setup: 50-dimensional quadratic with kappa = 500
L, mu = 500.0, 1.0
n = 50
Q = np.linspace(mu, L, n)  # eigenvalues uniformly in [mu, L]
x0 = np.ones(n)
num_iters = 500

# Run both methods
f_gd = gradient_descent(Q, x0, num_iters)
f_nag = nesterov_ag(Q, x0, num_iters)

# Theoretical upper bounds
R2 = np.sum(x0**2)
k_arr = np.arange(1, num_iters + 1)
gd_bound = L * R2 / (2 * k_arr)                   # O(1/k)
nag_bound = 2 * L * R2 / (k_arr + 1)**2           # O(1/k^2)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(f_gd[1:], color=TERRACOTTA, linewidth=2, label='Gradient Descent')
ax.semilogy(np.maximum(np.abs(f_nag[1:]), 1e-16), color=STEEL_BLUE, linewidth=2,
            label="Nesterov's AGD")
ax.semilogy(k_arr, gd_bound, '--', color=TERRACOTTA, linewidth=1.2, alpha=0.6,
            label='$O(1/k)$ bound')
ax.semilogy(k_arr, nag_bound, '--', color=STEEL_BLUE, linewidth=1.2, alpha=0.6,
            label='$O(1/k^2)$ bound')

ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x^k) - f(x^*)$')
ax.set_title(f'GD vs. Nesterov AGD on Quadratic ($n={n}$, $\\kappa = {int(L/mu)}$)')
ax.legend(fontsize=11)
ax.set_xlim([1, num_iters])
plt.tight_layout()
plt.show()

print(f'After {num_iters} iterations:')
print(f'  GD suboptimality:  {f_gd[-1]:.2e}')
print(f'  NAG suboptimality: {np.abs(f_nag[-1]):.2e}')

**Interpretation.** With $\kappa = 500$, the difference between GD and NAG is dramatic. GD barely makes progress over 500 iterations (the $O(1/k)$ rate is extremely slow when $L$ is large), while NAG converges at $O(1/k^2)$ --- orders of magnitude faster. The oscillations in NAG are characteristic: unlike GD, the accelerated method does *not* guarantee monotone decrease of $f$. The dashed lines confirm that both methods tightly match their theoretical rates.

## 2. FISTA for Lasso

FISTA (Fast Iterative Shrinkage-Thresholding Algorithm) extends Nesterov's acceleration to **composite** problems $\min\; g(x) + h(x)$, where $g$ is smooth and $h$ has a cheap proximal operator. For the Lasso problem
$$
\min_{x \in \mathbb{R}^n}\; \frac{1}{2}\|Ax - b\|_2^2 + \lambda\|x\|_1,
$$
we have $g(x) = \frac{1}{2}\|Ax - b\|_2^2$ (with $L = \|A^\top A\|$) and $h(x) = \lambda\|x\|_1$, whose proximal operator is **soft-thresholding**:
$$
\operatorname{prox}_{\alpha h}(z)_j = \operatorname{sign}(z_j)\max\{|z_j| - \alpha\lambda,\, 0\}.
$$

**FISTA** uses the same $\theta_k$ recursion as Nesterov's AGD:
$$
\begin{aligned}
x^k &= \mathcal{S}_{\lambda/L}\!\left(y^k - \frac{1}{L}A^\top(Ay^k - b)\right), \\
\theta_{k+1} &= \frac{1 + \sqrt{1 + 4\theta_k^2}}{2}, \\
y^{k+1} &= x^k + \frac{\theta_k - 1}{\theta_{k+1}}(x^k - x^{k-1}).
\end{aligned}
$$

FISTA converges at $O(1/k^2)$ versus ISTA's $O(1/k)$, with the same per-iteration cost.

In [ ]:
def soft_threshold(z, threshold):
    """Soft-thresholding (proximal operator of the l1-norm)."""
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0.0)


def lasso_objective(A, b, lam, x):
    """Compute 0.5*||Ax - b||^2 + lam*||x||_1."""
    return 0.5 * np.linalg.norm(A @ x - b)**2 + lam * np.linalg.norm(x, 1)


def ista_lasso(A, b, lam, L, x0, num_iters):
    """ISTA (proximal gradient) for LASSO."""
    x = x0.copy()
    obj_vals = [lasso_objective(A, b, lam, x)]
    for _ in range(num_iters):
        grad_g = A.T @ (A @ x - b)
        x = soft_threshold(x - grad_g / L, lam / L)
        obj_vals.append(lasso_objective(A, b, lam, x))
    return x, np.array(obj_vals)


def fista_lasso(A, b, lam, L, x0, num_iters):
    """FISTA (accelerated proximal gradient) for LASSO using exact theta_k recursion."""
    x_prev = x0.copy()
    x_curr = x0.copy()
    theta = 1.0
    obj_vals = [lasso_objective(A, b, lam, x_curr)]
    for _ in range(num_iters):
        # Momentum extrapolation
        theta_new = (1 + np.sqrt(1 + 4 * theta**2)) / 2
        momentum = (theta - 1) / theta_new
        y = x_curr + momentum * (x_curr - x_prev)
        # Proximal-gradient step
        grad_g = A.T @ (A @ y - b)
        x_new = soft_threshold(y - grad_g / L, lam / L)
        # Update
        x_prev = x_curr
        x_curr = x_new
        theta = theta_new
        obj_vals.append(lasso_objective(A, b, lam, x_curr))
    return x_curr, np.array(obj_vals)

In [ ]:
# Generate a Lasso problem with a sparse ground truth
np.random.seed(42)
m, n = 150, 300  # m samples, n features (underdetermined)
A = np.random.randn(m, n) / np.sqrt(m)

# Sparse ground truth with 15 nonzero entries
x_true = np.zeros(n)
support = np.random.choice(n, size=15, replace=False)
x_true[support] = np.random.randn(15)
b = A @ x_true + 0.05 * np.random.randn(m)

# Smoothness constant and regularization
L_lasso = np.linalg.norm(A.T @ A, 2)  # largest eigenvalue of A^T A
lam = 0.1 * np.max(np.abs(A.T @ b))   # standard choice

# Run ISTA and FISTA
x0 = np.zeros(n)
num_iters = 500
x_ista, obj_ista = ista_lasso(A, b, lam, L_lasso, x0, num_iters)
x_fista, obj_fista = fista_lasso(A, b, lam, L_lasso, x0, num_iters)

# Use FISTA solution as approximate optimum
f_star = min(obj_ista[-1], obj_fista[-1])

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 6))
iters = np.arange(num_iters + 1)
ax.semilogy(iters, np.maximum(obj_ista - f_star, 1e-16),
            color=TERRACOTTA, linewidth=2, label='ISTA (Proximal GD)')
ax.semilogy(iters, np.maximum(obj_fista - f_star, 1e-16),
            color=STEEL_BLUE, linewidth=2, label='FISTA (Accelerated)')

ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$F(x^k) - F(x^*)$')
ax.set_title('ISTA vs. FISTA on Lasso ($m=150$, $n=300$, 15 nonzeros)')
ax.legend(fontsize=12)
ax.set_xlim([0, num_iters])
plt.tight_layout()
plt.show()

print(f'ISTA final suboptimality:  {obj_ista[-1] - f_star:.2e}')
print(f'FISTA final suboptimality: {obj_fista[-1] - f_star:.2e}')

**Interpretation.** FISTA converges dramatically faster than ISTA on this Lasso problem. Both methods perform the same per-iteration work (one matrix-vector product with $A$ and $A^\top$, plus soft-thresholding), but the momentum extrapolation in FISTA yields an $O(1/k^2)$ rate versus ISTA's $O(1/k)$. Notice the characteristic oscillations in FISTA --- the objective is not monotonically decreasing, which is a hallmark of accelerated methods.

## 3. Momentum Visualization

To build geometric intuition, we plot the **iterate trajectories** of GD vs. NAG on a 2D ill-conditioned quadratic:
$$
f(x_1, x_2) = \frac{1}{2}(L\, x_1^2 + \mu\, x_2^2),
$$
with $L = 50$, $\mu = 1$ ($\kappa = 50$). The level sets are elongated ellipses. GD with stepsize $1/L$ oscillates back and forth across the narrow valley (the high-curvature $x_1$ direction), while NAG uses momentum to damp these oscillations and make faster progress along the low-curvature $x_2$ direction.

In [ ]:
def gd_trajectory(Q, x0, num_iters):
    """Return full iterate trajectory for gradient descent."""
    L = np.max(Q)
    x = x0.copy()
    traj = [x.copy()]
    for _ in range(num_iters):
        grad = Q * x
        x = x - (1.0 / L) * grad
        traj.append(x.copy())
    return np.array(traj)


def nag_trajectory(Q, x0, num_iters):
    """Return full iterate trajectory for Nesterov's AGD (exact theta_k)."""
    L = np.max(Q)
    x = x0.copy()
    y = x0.copy()
    theta = 1.0
    traj = [x.copy()]
    for k in range(num_iters):
        grad = Q * x
        y_new = x - (1.0 / L) * grad
        theta_new = (1 + np.sqrt(1 + 4 * theta**2)) / 2
        momentum = (theta - 1) / theta_new
        x_new = y_new + momentum * (y_new - y)
        y = y_new
        x = x_new
        theta = theta_new
        traj.append(x.copy())
    return np.array(traj)


# Setup
L, mu = 50.0, 1.0
Q = np.array([L, mu])
x0 = np.array([1.0, 7.0])
num_iters = 80

traj_gd = gd_trajectory(Q, x0, num_iters)
traj_nag = nag_trajectory(Q, x0, num_iters)

# Contour plot
x1 = np.linspace(-1.5, 1.5, 300)
x2 = np.linspace(-8, 8, 300)
X1, X2 = np.meshgrid(x1, x2)
F = 0.5 * (L * X1**2 + mu * X2**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, traj, title, color in zip(
    axes,
    [traj_gd, traj_nag],
    ['Gradient Descent', "Nesterov's AGD"],
    [TERRACOTTA, STEEL_BLUE]
):
    levels = [5, 15, 30, 50, 80, 120, 180]
    ax.contour(X1, X2, F, levels=levels, colors='#d5cdc4', linewidths=0.8)
    ax.plot(traj[:, 0], traj[:, 1], 'o-', color=color, markersize=2.5,
            linewidth=1.2, label=title, alpha=0.9)
    ax.plot(traj[0, 0], traj[0, 1], 's', color=GOLD, markersize=10,
            zorder=5, label='Start')
    ax.plot(0, 0, '*', color=SAGE, markersize=15, zorder=5, label='$x^*$')
    ax.set_xlabel('$x_1$')
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10, loc='upper right')
    ax.set_aspect('auto')
    ax.set_xlim([-1.5, 1.5])

axes[0].set_ylabel('$x_2$')
fig.suptitle(f'Iterate Trajectories on Ill-Conditioned Quadratic ($\\kappa = {int(L/mu)}$)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation.** The two panels show a striking contrast. GD oscillates heavily in the $x_1$ direction (high curvature, eigenvalue $L=50$) while making slow progress along $x_2$ (low curvature, eigenvalue $\mu=1$). The stepsize $1/L$ is conservative for the $x_2$ direction but necessary to avoid divergence in $x_1$. NAG initially overshoots (a consequence of momentum), but then quickly corrects and navigates the valley much more efficiently. The momentum term accumulates information about the low-curvature direction and effectively increases the stepsize in that direction.

## 4. Strongly Convex Case: GD vs. Heavy-Ball vs. Nesterov AGD

When $f$ is additionally $\mu$-strongly convex, we can exploit this structure to get **linear** convergence with improved $\kappa$ dependence:

| Method | Rate | Requires |
|---|---|---|
| GD | $(1 - 1/\kappa)^k$ | $L$ |
| Heavy-ball | $\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^{2k}$ | $L$, $\mu$ |
| Nesterov AGD-SC | $\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^{2k}$ | $L$, $\mu$ |

The acceleration gap is $\sqrt{\kappa}$ --- both heavy-ball and Nesterov-SC are dramatically faster than GD on ill-conditioned problems.

**Heavy-ball** (Polyak, 1964):
$$
x^{k+1} = x^k - \eta\nabla f(x^k) + \theta(x^k - x^{k-1}), \quad \eta = \frac{4}{(\sqrt{L}+\sqrt{\mu})^2},\; \theta = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2.
$$

**Nesterov AGD-SC:** Fixed momentum $\beta = (\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)$.

We compare all three on a 50-dimensional quadratic with $\kappa = 500$.

In [ ]:
def heavy_ball(Q, x0, num_iters):
    """Heavy-ball method with optimal parameters for quadratic."""
    L = np.max(Q)
    mu = np.min(Q)
    kappa = L / mu
    eta = 4.0 / (np.sqrt(L) + np.sqrt(mu))**2
    beta = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1))**2
    x = x0.copy()
    x_prev = x0.copy()
    f_vals = [0.5 * x @ (Q * x)]
    for _ in range(num_iters):
        grad = Q * x
        x_new = x - eta * grad + beta * (x - x_prev)
        x_prev = x.copy()
        x = x_new
        f_vals.append(0.5 * x @ (Q * x))
    return np.array(f_vals)


def nesterov_ag_sc(Q, x0, num_iters):
    """Nesterov's AGD for strongly convex functions (fixed momentum)."""
    L = np.max(Q)
    mu = np.min(Q)
    gamma = np.sqrt(mu / L)
    beta = (1 - gamma) / (1 + gamma)
    x = x0.copy()
    y = x0.copy()
    f_vals = [0.5 * y @ (Q * y)]
    for _ in range(num_iters):
        grad = Q * x
        y_new = x - (1.0 / L) * grad
        x_new = y_new + beta * (y_new - y)
        y = y_new
        x = x_new
        f_vals.append(0.5 * y @ (Q * y))
    return np.array(f_vals)


# Problem: 50-dimensional quadratic with kappa = 500
L, mu = 500.0, 1.0
kappa = L / mu
n = 50
Q = np.linspace(mu, L, n)
x0 = np.ones(n)
num_iters = 300

f_gd = gradient_descent(Q, x0, num_iters)
f_hb = heavy_ball(Q, x0, num_iters)
f_sc = nesterov_ag_sc(Q, x0, num_iters)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
iters = np.arange(num_iters + 1)

ax.semilogy(iters, np.maximum(f_gd, 1e-16), color=TERRACOTTA, linewidth=2,
            label=f'GD (step 1/L)')
ax.semilogy(iters, np.maximum(np.abs(f_hb), 1e-16), color=SAGE, linewidth=2,
            label=f'Heavy-ball (optimal $\\eta$, $\\theta$)')
ax.semilogy(iters, np.maximum(np.abs(f_sc), 1e-16), color=STEEL_BLUE, linewidth=2,
            label=f'Nesterov AGD-SC ($\\beta = (\\sqrt{{\\kappa}}-1)/(\\sqrt{{\\kappa}}+1)$)')

ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x^k) - f(x^*)$')
ax.set_title(f'Strongly Convex: GD vs. Heavy-Ball vs. Nesterov AGD ($n={n}$, $\\kappa = {int(kappa)}$)')
ax.legend(fontsize=11)
ax.set_xlim([0, num_iters])
plt.tight_layout()
plt.show()

print(f'After {num_iters} iterations:')
print(f'  GD:           {f_gd[-1]:.2e}')
print(f'  Heavy-ball:   {np.abs(f_hb[-1]):.2e}')
print(f'  Nesterov-SC:  {np.abs(f_sc[-1]):.2e}')

In [ ]:
# Condition number sweep: iterations to epsilon
np.random.seed(0)
dim = 20
kappas = np.array([5, 10, 20, 50, 100, 200, 500, 1000])
eps = 1e-6
n_trials = 5

def iters_to_eps(method_fn, Q, x0, eps, max_iters=50000):
    """Count iterations for a method to reach f(x) <= eps."""
    f_vals = method_fn(Q, x0, max_iters)
    below = np.where(np.abs(f_vals) <= eps)[0]
    return below[0] if len(below) > 0 else max_iters

gd_counts, hb_counts, sc_counts = [], [], []

for kappa in kappas:
    gd_t, hb_t, sc_t = [], [], []
    for trial in range(n_trials):
        eigvals = np.linspace(1.0, kappa, dim).astype(float)
        x0 = np.random.randn(dim)
        x0 = x0 / np.linalg.norm(x0) * 5
        gd_t.append(iters_to_eps(gradient_descent, eigvals, x0, eps))
        hb_t.append(iters_to_eps(heavy_ball, eigvals, x0, eps))
        sc_t.append(iters_to_eps(nesterov_ag_sc, eigvals, x0, eps))
    gd_counts.append(np.mean(gd_t))
    hb_counts.append(np.mean(hb_t))
    sc_counts.append(np.mean(sc_t))

gd_counts = np.array(gd_counts)
hb_counts = np.array(hb_counts)
sc_counts = np.array(sc_counts)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(kappas, gd_counts, 'o-', color=TERRACOTTA, linewidth=2, markersize=8, label='GD')
ax.loglog(kappas, hb_counts, 's-', color=SAGE, linewidth=2, markersize=8, label='Heavy-ball')
ax.loglog(kappas, sc_counts, '^-', color=STEEL_BLUE, linewidth=2, markersize=8, label='Nesterov AGD-SC')

# Reference lines
c_gd = gd_counts[3] / kappas[3]
c_sc = sc_counts[3] / np.sqrt(kappas[3])
ax.loglog(kappas, c_gd * kappas, '--', color=TERRACOTTA, linewidth=1, alpha=0.5, label='$O(\\kappa)$')
ax.loglog(kappas, c_sc * np.sqrt(kappas), '--', color=STEEL_BLUE, linewidth=1, alpha=0.5, label='$O(\\sqrt{\\kappa})$')

ax.set_xlabel('Condition number $\\kappa$')
ax.set_ylabel('Iterations to $\\varepsilon = 10^{-6}$')
ax.set_title('Condition Number Dependence: $O(\\kappa)$ vs. $O(\\sqrt{\\kappa})$')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'At kappa=1000:  GD ~{gd_counts[-1]:.0f},  HB ~{hb_counts[-1]:.0f},  NAG-SC ~{sc_counts[-1]:.0f}')
print(f'Speedup (GD/NAG-SC): {gd_counts[-1]/sc_counts[-1]:.1f}x')

**Interpretation.** On the log-log plot, GD follows the $O(\kappa)$ reference line (slope $\approx 1$), while both heavy-ball and Nesterov AGD-SC follow the $O(\sqrt{\kappa})$ reference (slope $\approx 0.5$). The gap between GD and the accelerated methods grows as $\sqrt{\kappa}$, confirming the theoretical prediction. At $\kappa = 1000$, the accelerated methods need roughly $\sqrt{1000} \approx 32$ times fewer iterations than GD.

## 5. Adaptive Restart

The convex-case NAG has a fundamental limitation: its momentum $\beta_k = (\theta_k - 1)/\theta_{k+1} \to 1$ grows monotonically, eventually causing oscillations when the function is actually strongly convex. The SC variant fixes this but requires knowing $\mu$.

**Gradient restart** (O'Donoghue and Candes, 2015) resolves this: run convex-case NAG, but **reset $\theta \leftarrow 1$** whenever the momentum direction points uphill:
$$
\text{Restart when:} \quad \langle \nabla f(y^{k+1}),\, x^{k+1} - x^k \rangle > 0.
$$

**Why this works:** The vector $x^{k+1} - x^k$ is the extrapolation (momentum) step. When its inner product with $\nabla f$ is positive, the momentum is pushing *uphill* — the method has overshot the valley floor. Resetting $\theta \leftarrow 1$ kills the momentum and starts a fresh AGD phase.

**Why this gives linear convergence:** Each fresh AGD phase gives $O(1/k^2)$ convergence within that phase. On a $\mu$-strongly convex function, overshoot is detected after $O(\sqrt{\kappa})$ iterations, so each phase reduces the gap by a constant factor. Compounding these phases gives geometric (linear) convergence at rate $O(\sqrt{\kappa} \log(1/\varepsilon))$ — the same as the SC variant, but without knowing $\mu$.

In [ ]:
def nag_with_restart(Q, x0, num_iters):
    """NAG (convex case, exact theta_k) with gradient-based adaptive restart.

    At each step:
    1. Compute gradient step: y_new = x - (1/L)*grad
    2. Compute theta_new and momentum coefficient
    3. Extrapolate: x_new = y_new + mom*(y_new - y)
    4. Check restart: if <grad_f(y_new), x_new - x> > 0, reset theta=1
    """
    L = np.max(Q)
    x = x0.copy()
    y = x0.copy()
    theta = 1.0
    f_vals = [0.5 * np.sum(Q * y**2)]
    restart_iters = []

    for k in range(num_iters):
        grad = Q * x
        y_new = x - (1.0 / L) * grad
        theta_new = (1 + np.sqrt(1 + 4 * theta**2)) / 2
        momentum = (theta - 1) / theta_new
        x_new = y_new + momentum * (y_new - y)

        # Gradient restart check: is momentum pushing uphill?
        grad_at_y = Q * y_new                # nabla f(y^{k+1})
        mom_dir = x_new - x                  # extrapolation step x^{k+1} - x^k
        if np.dot(grad_at_y, mom_dir) > 0 and theta > 1.5:
            # Overshoot detected: reset momentum
            theta = 1.0
            x = y_new.copy()
            y = y_new.copy()
            restart_iters.append(k)
        else:
            x = x_new
            y = y_new
            theta = theta_new

        f_vals.append(0.5 * np.sum(Q * y**2))

    return np.array(f_vals), restart_iters


def nag_no_restart(Q, x0, num_iters):
    """NAG (convex case, exact theta_k) without restart."""
    L = np.max(Q)
    x = x0.copy()
    y = x0.copy()
    theta = 1.0
    f_vals = [0.5 * np.sum(Q * y**2)]
    for k in range(num_iters):
        grad = Q * x
        y_new = x - (1.0 / L) * grad
        theta_new = (1 + np.sqrt(1 + 4 * theta**2)) / 2
        momentum = (theta - 1) / theta_new
        x_new = y_new + momentum * (y_new - y)
        y = y_new
        x = x_new
        theta = theta_new
        f_vals.append(0.5 * np.sum(Q * y**2))
    return np.array(f_vals)


def gd_values(Q, x0, num_iters):
    """GD function values for reference."""
    L = np.max(Q)
    x = x0.copy()
    f_vals = [0.5 * np.sum(Q * x**2)]
    for _ in range(num_iters):
        x = x - (1.0 / L) * (Q * x)
        f_vals.append(0.5 * np.sum(Q * x**2))
    return np.array(f_vals)

In [ ]:
# Strongly convex quadratic with kappa = 100
L, mu = 100.0, 1.0
Q = np.linspace(mu, L, 30)
x0 = np.ones(len(Q)) * 3.0
num_iters = 500

f_gd = gd_values(Q, x0, num_iters)
f_nag_no = nag_no_restart(Q, x0, num_iters)
f_nag_re, restarts = nag_with_restart(Q, x0, num_iters)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
iters = np.arange(num_iters + 1)

ax.semilogy(iters, np.maximum(f_gd, 1e-16), color=GOLD, linewidth=1.5,
            linestyle=':', label='GD', alpha=0.8)
ax.semilogy(iters, np.maximum(np.abs(f_nag_no), 1e-16), color=TERRACOTTA,
            linewidth=2, label='NAG (no restart)')
ax.semilogy(iters, np.maximum(np.abs(f_nag_re), 1e-16), color=STEEL_BLUE,
            linewidth=2, label='NAG + gradient restart')

# Mark restart points
for ri in restarts:
    ax.axvline(ri, color=LAVENDER, alpha=0.2, linewidth=0.8)
if restarts:
    ax.axvline(restarts[0], color=LAVENDER, alpha=0.2, linewidth=0.8,
               label='Restart events')

ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x^k) - f(x^*)$')
ax.set_title(f'Effect of Gradient Restart ($\\kappa = {int(L/mu)}$, $d={len(Q)}$)')
ax.legend(fontsize=11)
ax.set_xlim([0, num_iters])
plt.tight_layout()
plt.show()

print(f'Number of restarts: {len(restarts)}')
print(f'Final suboptimality:')
print(f'  GD:            {f_gd[-1]:.2e}')
print(f'  NAG (no rst):  {np.abs(f_nag_no[-1]):.2e}')
print(f'  NAG + restart: {np.abs(f_nag_re[-1]):.2e}')

**Interpretation.** On this strongly convex problem ($\kappa = 100$), the three methods show strikingly different behaviors:

- **GD** (dotted gold) converges linearly at rate $(1 - 1/\kappa)^k$, which is slow for large $\kappa$.
- **NAG without restart** (red) converges at $O(1/k^2)$ but does not exploit strong convexity. The oscillations are visible.
- **NAG with gradient restart** (blue) achieves the best of both worlds: it uses the convex-case $\theta_k$ momentum schedule but adaptively resets momentum when overshoot is detected (purple lines). This converts the $O(1/k^2)$ polynomial rate into a *linear* rate, recovering the optimal $O(\sqrt{\kappa}\log(1/\varepsilon))$ complexity without requiring knowledge of $\mu$.